In [1]:
import os
import cv2
import csv
import torch
import glob
import numpy as np
import pandas as pd
from tqdm import tqdm
import ultralytics
import plotly
import plotly.express as px
import plotly.graph_objects as go
import tkinter as tk
from tkinter import font
import seaborn as sns
from tkinter import filedialog
from PIL import Image,ImageTk
from skimage.draw import polygon
from skimage.measure import regionprops
from skimage import filters, measure, morphology
from skimage import measure
from ultralytics import YOLO
from pathlib import Path
from ultralytics import SAM
import matplotlib.pyplot as plt
from ultralytics.data import YOLODataset
from ultralytics.utils import LOGGER
from ultralytics.utils.ops import xywh2xyxy
from IPython.display import Image as IPyImage


### GUI 

In [ ]:

best_model = r'D:\Code\boutRight_yolo\runs\detect\train2\weights\best.pt'
model = YOLO(best_model)

def predict_image(image_path):
    results = model.predict(image_path, save=False, imgsz=320, conf=0.4, show_conf=False, show_labels=False)
    return results

def draw_boxes(img, boxes):
    for box in boxes:
        cls = int(box.cls)  
        conf = float(box.conf)
        xywh = box.xywh.cpu().numpy().flatten()
        x, y, w, h = xywh

        img_height, img_width, _ = img.shape
        l = int(x - w / 2)
        r = int(x + w / 2)
        t = int(y - h / 2)
        b = int(y + h / 2)

        color_mapping = {
            (255, 0, 0): (0, 0, 255), 
            (255, 255, 0): (0, 255, 255)
        }

        original_colors = [(255, 255, 0), (0, 255, 0), (255, 0, 0), (255, 255, 0)]
    
        color = original_colors[cls]
        color = color_mapping.get(color, color) 

        cv2.rectangle(img, (l, t), (r, b), color, 2)
    return img

def select_image():
    filepath = filedialog.askopenfilename(filetypes=[("Image Files", "*.jpg;*.png")])
    if filepath:
        global image_path, uploaded_image
        image_path = filepath
        uploaded_image = cv2.imread(filepath)
        img_height, img_width, _ = uploaded_image.shape
        canvas.config(width=img_width, height=img_height)
        display_image(uploaded_image)


def segment_image():
    global image_path
    img = cv2.imread(image_path)
    results = predict_image(image_path)
    img_with_boxes = draw_boxes(img.copy(), results[0].boxes)
    display_image(img_with_boxes)


def display_image(img):
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_pil = Image.fromarray(img_rgb)
    img_tk = ImageTk.PhotoImage(image=img_pil)
    canvas.create_image(0, 0, anchor=tk.NW, image=img_tk)
    canvas.image = img_tk  


window = tk.Tk()
window.title("Image Segmentation")

canvas = tk.Canvas(window)
canvas.pack()

button_font = font.Font(family="Helvetica", size=12, weight="bold")

button_select = tk.Button(window, text="Select Image", command=select_image, font=button_font, bg='#f24f26', fg="white", padx=20, pady=10, relief="raised", bd=3)
button_select.pack()

button_segment = tk.Button(window, text="Run Inference", command=segment_image, font=button_font, bg='#f24f26', fg="white", padx=20, pady=10, relief="raised", bd=3)
button_segment.pack()

window.mainloop()


## Run inference on test dataset and save the overlay 

In [ ]:
model_path = r'D:\Code\boutRight_yolo\runs\detect\train2\weights\best.pt'
img_fol = r'D:\Code\boutRight_yolo\dataset\test\images' 
label_dir = r'D:\Code\boutRight_yolo\dataset\test\model_labels2'

model = YOLO(model=model_path)

for img_name in os.listdir(img_fol):
    if img_name.lower().endswith(('jpg', 'jpeg', 'png')):  
        img_path = os.path.join(img_fol, img_name)
        
        results = model.predict(img_path, save=True, conf=0.4, show_conf=False, show_labels=False)
        
        bb = []
        boxes = results[0].boxes
        img = cv2.imread(img_path)
        img_height, img_width, _ = img.shape  

        for box in boxes:
            cls = int(box.cls)  
            confid = float(box.conf) 

            xywh = box.xywh.cpu().numpy().flatten()
            x, y, w, h = xywh  

            x_center_norm = x / img_width
            y_center_norm = y / img_height
            w_norm = w / img_width
            h_norm = h / img_height

            bb.append([cls, x_center_norm, y_center_norm, w_norm, h_norm])

        label_f = os.path.join(label_dir, os.path.splitext(img_name)[0] + '.txt')
        os.makedirs(label_dir, exist_ok=True)  
        
        with open(label_f, 'w') as label_file:
            for entry in bb:
                label_file.write(f"{entry[0]} {entry[1]} {entry[2]} {entry[3]} {entry[4]}\n")

## Original and model prediction plot side by side

In [ ]:

def draw_boxes(img, label_p):
    with open(label_p, 'r') as fl:
        data = fl.readlines()
    
    dh, dw, _ = img.shape

    for dt in data:
        cls, x, y, w, h = map(float, dt.split(' '))
        l = int((x - w / 2) * dw)
        r = int((x + w / 2) * dw)  
        t = int((y - h / 2) * dh)  
        b = int((y + h / 2) * dh)

        color = [(255, 255, 0), (0, 255, 0), (255, 0, 0), (255, 255, 0)][int(cls) % 4]
        cv2.rectangle(img, (l, t), (r, b), color, 2)

    return img

def overlay_ann(img_folder, label_folder, model_label_fol, output_folder):
    os.makedirs(output_folder, exist_ok=True)
    image_files = [f for f in os.listdir(img_folder) if f.endswith('.jpg')]

    for img_file in image_files:
        img_p = os.path.join(img_folder, img_file)
        label_p = os.path.join(label_folder, os.path.splitext(img_file)[0] + '.txt')
        model_label_p = os.path.join(model_label_fol, os.path.splitext(img_file)[0] + '.txt')

        img = cv2.imread(img_p)
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        img_original = img_rgb.copy()
        if os.path.exists(label_p):
            img_original = draw_boxes(img_original, label_p)
        img_model = img_rgb.copy()
        if os.path.exists(model_label_p):
            img_model = draw_boxes(img_model, model_label_p)

            save_path = os.path.join(output_folder, img_file)
            img_bgr = cv2.cvtColor(img_model, cv2.COLOR_RGB2BGR)
            cv2.imwrite(save_path, img_bgr)
        fig, axes = plt.subplots(1, 2, figsize=(12, 6))
        axes[0].imshow(img_original)
        axes[0].set_title('Original Labels')
        axes[0].axis('off')

        axes[1].imshow(img_model)
        axes[1].set_title('Model Labels')
        axes[1].axis('off')
        plt.tight_layout()
        plt.show()

img_folder = r'D:\Code\boutRight_yolo\dataset\test\images'
label_folder = r'D:\Code\boutRight_yolo\dataset\test\labels'
model_label_fol = r'D:\Code\boutRight_yolo\dataset\test\model_labels2'
output_folder = r'D:\Code\boutRight_yolo\dataset\test\model_output_images2'

overlay_ann(img_folder, label_folder, model_label_fol, output_folder)
